In [0]:
!pip install openpyxl
!pip install polars fastexcel
dbutils.library.restartPython() 

In [0]:
import pandas as pd
import polars as pl
import fastexcel
import pickle
import numpy as np
import os
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
import calendar
# for moving between compute and volume
import shutil

from datetime import date
warnings.filterwarnings('ignore')

In [0]:
bcg_path = '/Volumes/opsanalytics_adb_workspace01/or/mappings_and_raw_data'
output_path = '/Volumes/opsanalytics_adb_workspace01/or/model_outputs'
or_case_table = 'opsanalytics_adb_workspace01.or.or_quality_dashboard_case_details'
holidays_table = 'opsanalytics_adb_workspace01.lab.mshs_holidays'

In [0]:
cpt_cardio_ortho_data = pd.read_excel(f'{bcg_path}/OAO Update 20260820_IP_OP.xlsx',sheet_name='CPT_Mapping',skiprows=3)[['CPT_Code','Service_Line','Sub_Service','Type']]
cpt_cardio_ortho_data['CPT_Code'] = cpt_cardio_ortho_data['CPT_Code'].str.strip()
cpt_cardio_ortho_cpt_codes = pd.read_excel(f'{bcg_path}/OAO Update 20260820_IP_OP.xlsx',sheet_name='CPT_Mapping',skiprows=3)['CPT_Code'].str.strip()
or_schedules = pd.read_excel(f'{bcg_path}/ORSchedules.xlsx')
or_schedules = or_schedules.loc[or_schedules.groupby('Location')["# ORs"].idxmax()]
or_schedules.rename(columns={'Location':'HOSPITAL'},inplace=True)
# or_schedules.to_dict()
# or_schedules = spark.createDataFrame(or_schedules)

In [0]:
def getPrimeTimeStart(SITE):
    return or_schedules.loc[or_schedules['HOSPITAL']==SITE]['Time Start'].values[0]
def getPrimeTimeEnd(SITE):
    return or_schedules.loc[or_schedules['HOSPITAL']==SITE]['Time End'].values[0]

In [0]:
display(cpt_cardio_ortho_data)

In [0]:
from pyspark.sql import functions as F

# ------------------------------------------------------------------
# Ensure required variables are defined (fallback to the literal paths)
# ------------------------------------------------------------------
or_case_table = globals().get(
    "or_case_table",
    "opsanalytics_adb_workspace01.or.or_quality_dashboard_case_details"
)

holidays_table = globals().get(
    "holidays_table",
    "opsanalytics_adb_workspace01.lab.mshs_holidays"
)

# 1. Load the OR Case Table natively (replaces the SQL and .toPandas())
df_spark = spark.sql(
    f"""
    select OR_LOCATION,OR_CASE_ID, SURGERY_DATE,PATIENT_OUT_ROOM_DTTM,PATIENT_IN_ROOM_DTTM,PRIMARY_PROC_CODE, SURGEON_NPI,PRIMARY_SURGEON,
        CASE 
                WHEN HOSPITAL IS NULL AND  OR_LOCATION LIKE '%MSSN%' THEN 'MSSN' 
                ELSE HOSPITAL
        END AS HOSPITAL
    from opsanalytics_adb_workspace01.or.or_quality_dashboard_case_details
    where PATIENT_IN_ROOM_DTTM IS NOT NULL AND
        (year(SURGERY_DATE) = 2025 
            OR SURGERY_DATE BETWEEN '2026-01-01' AND '2026-06-30');
    """
)

# 2. Load the Holidays Table as a pandas DataFrame (not pandas‑on‑Spark)
holidays_df = spark.table(holidays_table).toPandas()
holidays_df['holiday_date'] = holidays_df['holiday_date'].astype('datetime64[ns]')


my_columns = ["HOSPITAL","OR_LOCATION","OR_CASE_ID", "SURGERY_DATE","PATIENT_OUT_ROOM_DTTM","PATIENT_IN_ROOM_DTTM","PRIMARY_PROC_CODE","SURGEON_NPI","PRIMARY_SURGEON"] 

# Convert to a pandas DataFrame for subsequent pandas‑only operations
df = df_spark.select(*my_columns).toPandas()

# Cleaning CPT and keeping only matched CPT
df['PRIMARY_PROC_CODE'] = df['PRIMARY_PROC_CODE'].astype(str).str.strip()
df = df.loc[df['PRIMARY_PROC_CODE'].isin(cpt_cardio_ortho_cpt_codes)]
# ------------------------------------------------------------------
# Add the PrimeTimeStart and PrimeTimeEnd columns


# Merge with the schedules (already a pandas DataFrame)
df['PrimeTimeStart'] = df['HOSPITAL'].apply(getPrimeTimeStart)
df['PrimeTimeEnd']   = df['HOSPITAL'].apply(getPrimeTimeEnd)


# 1. Clean the date column to remove the trailing 'T00:00:00...' zone data
df['SURGERY_DATE'] = pd.to_datetime(df['SURGERY_DATE']).dt.date

# 2. Combine the Date with the PrimeTimeStart and PrimeTimeEnd text fields
df['PrimeTimeStart_dt'] = pd.to_datetime(df['SURGERY_DATE'].astype(str) + ' ' + df['PrimeTimeStart'])
df['PrimeTimeEnd_dt'] = pd.to_datetime(df['SURGERY_DATE'].astype(str) + ' ' + df['PrimeTimeEnd'])
df['SURGERY_DATE'] = pd.to_datetime(df['SURGERY_DATE'])

# 4. Create the boolean flag indicating if it happened in prime time
df['is_prime_time'] = (df['PATIENT_IN_ROOM_DTTM'] <= df['PrimeTimeEnd_dt'])

# Holidays and Weekends
df['IsWeekend'] = np.where(df['SURGERY_DATE'].dt.dayofweek.isin([5,6]),'Yes','No')
df['IsHoliday'] = np.where(df['SURGERY_DATE'].isin(holidays_df['holiday_date'].to_list()),'Yes','No')


# Case Minutes and Year
df['case_minutes'] = (df['PATIENT_OUT_ROOM_DTTM'] - df['PATIENT_IN_ROOM_DTTM']).dt.total_seconds() / 60
df['year'] = df['SURGERY_DATE'].dt.year

# Join with CPT Mapping List
df = df.merge(cpt_cardio_ortho_data, how='left', left_on='PRIMARY_PROC_CODE',right_on='CPT_Code')



In [0]:
df_select = df[['HOSPITAL',
                'SURGEON_NPI', 
                'PRIMARY_SURGEON',
                'OR_CASE_ID',
                'SURGERY_DATE',
                'year',
                'case_minutes',
                'CPT_Code',
                'Service_Line',
                'Sub_Service',
                'Type',
                'IsWeekend',
                'IsHoliday',
                'is_prime_time']]

In [0]:
local_path = '/tmp/CPTCaseRaw.xlsx'
with pd.ExcelWriter(local_path, engine='openpyxl') as writer:
        df_select.to_excel(writer, sheet_name='RawData',index=False)
        holidays_weekends_excluded.to_excel(writer, sheet_name='Holidays-Weekends Excluded',startrow=5, startcol=1,index=False)


shutil.move(local_path, output_path + '/CPTCaseRaw.xlsx')

In [0]:
cpt_missing_in_or_data = cpt_cardio_ortho_data.loc[~cpt_cardio_ortho_data['CPT_Code'].isin(df_select['CPT_Code'].unique())]

In [0]:
local_path = '/tmp/CPTCaseRaw.xlsx'
with pd.ExcelWriter(local_path, engine='openpyxl') as writer:
        df_select.to_excel(writer, sheet_name='RawData',index=False)
        cpt_missing_in_or_data.to_excel(writer, sheet_name='CPT Not In OR',startrow=5, startcol=1,index=False)


shutil.move(local_path, output_path + '/CPTCaseRaw.xlsx')

In [0]:
def getVolumeandCaseMinutes(data,exclude_holidays_weekends =True,exclude_non_primetime = True, grouping = 'SITE'):
    # get cases with listed CPTs
    df = data.loc[data['PRIMARY_PROC_CODE'].isin(cpt_cardio_ortho_cpt_codes)]
    if exclude_holidays_weekends:
        # Exclude Weekends
        df = df.loc[~df['SURGERY_DATE'].dt.dayofweek.isin([5,6])]
        # Exclude Holidays
        df = df.loc[~df['SURGERY_DATE'].isin(holidays_df['holiday_date'].to_list())]
    if exclude_non_primetime:
        df = df.loc[df['is_prime_time']]
    # This converts the timedelta object directly into a clean, floating-point number of minutes
    df['case_minutes'] = (df['PATIENT_OUT_ROOM_DTTM'] - df['PATIENT_IN_ROOM_DTTM']).dt.total_seconds() / 60
    df['year'] = df['SURGERY_DATE'].dt.year

    if grouping == 'SITE':
        total_case_minutes = df.groupby(["PRIMARY_PROC_CODE","year","HOSPITAL"])[['case_minutes']].sum().reset_index()
        case_volume = df.groupby(["PRIMARY_PROC_CODE","year","HOSPITAL"])[['OR_CASE_ID']].count().reset_index()

        aggregated =  pd.merge(case_volume,total_case_minutes,on=['PRIMARY_PROC_CODE','year','HOSPITAL'])
        aggregated['avg_case_minutes'] = aggregated['case_minutes']/aggregated['OR_CASE_ID']
        aggregated.rename(columns={'OR_CASE_ID':'case_volume',
                                'PRIMARY_PROC_CODE' : 'CPT_Code'},inplace=True)
        # aggregated = aggregated.pivot(index='CPT_Code', columns='year', values=['case_volume','case_minutes','avg_case_minutes']).fillna(0).reset_index()

        return aggregated
    else:
        total_case_minutes = df.groupby(["PRIMARY_PROC_CODE","year"])[['case_minutes']].sum().reset_index()
        case_volume = df.groupby(["PRIMARY_PROC_CODE","year"])[['OR_CASE_ID']].count().reset_index()

        aggregated =  pd.merge(case_volume,total_case_minutes,on=['PRIMARY_PROC_CODE','year'])
        aggregated['avg_case_minutes'] = aggregated['case_minutes']/aggregated['OR_CASE_ID']
        aggregated.rename(columns={'OR_CASE_ID':'case_volume',
                                'PRIMARY_PROC_CODE' : 'CPT_Code'},inplace=True)
        aggregated = aggregated.pivot(index='CPT_Code', columns='year', values=['case_volume','case_minutes','avg_case_minutes']).fillna(0).reset_index()

        return aggregated



In [0]:
no_exclusions = getVolumeandCaseMinutes(df,exclude_holidays_weekends=False,exclude_non_primetime=False,grouping='SITE')

In [0]:
no_exclusions

In [0]:
holidays_weekends_excluded = getVolumeandCaseMinutes(df,exclude_holidays_weekends=True,exclude_non_primetime=False)

In [0]:
display(holidays_weekends_excluded)

In [0]:
holidays_weekends_excluded_and_primetime_only = getVolumeandCaseMinutes(df,exclude_holidays_weekends=True,exclude_non_primetime=True)

In [0]:
display(holidays_weekends_excluded_and_primetime_only)

In [0]:
# 1. Flatten the MultiIndex columns for each DataFrame
no_exclusions.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) else col for col in no_exclusions.columns]
holidays_weekends_excluded.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) else col for col in holidays_weekends_excluded.columns]
holidays_weekends_excluded_and_primetime_only.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) else col for col in holidays_weekends_excluded_and_primetime_only.columns]


# Write locally to the temporary container folder first
local_path = '/tmp/CPTCaseVolume.xlsx'
with pd.ExcelWriter(local_path, engine='openpyxl') as writer:
        no_exclusions.to_excel(writer, sheet_name='No Exclusions',startrow=5, startcol=1, index=False)
        holidays_weekends_excluded.to_excel(writer, sheet_name='Holidays-Weekends Excluded',startrow=5, startcol=1,index=False)
        holidays_weekends_excluded_and_primetime_only.to_excel(writer, sheet_name='Holidays-Weekends-NPT-Excluded',startrow=5,startcol=1, index=False)

        # 2. Grab the sheet to inject your custom headers at the top
        workbook = writer.book
        worksheet = writer.sheets['No Exclusions']
        # 3. Add your custom data into the top rows
        worksheet['B2'] = "Excluded Holidays and Weekends"
        worksheet['C2'] = "No"
        worksheet['B3'] = "Included Non Prime Time"
        worksheet['C3'] = "Yes"
        
        worksheet = writer.sheets['Holidays-Weekends Excluded']
        worksheet['B2'] = "Excluded Holidays and Weekends"
        worksheet['C2'] = "Yes"
        worksheet['B3'] = "Included Non Prime Time"
        worksheet['C3'] = "Yes"
        
        worksheet = writer.sheets['Holidays-Weekends-NPT-Excluded']
        worksheet['B2'] = "Excluded Holidays and Weekends"
        worksheet['C2'] = "Yes"
        worksheet['B3'] = "Included Non Prime Time"
        worksheet['C3'] = "No"

# Move it to your final cloud/network path
shutil.move(local_path, output_path + '/CPTCaseVolume.xlsx')